In [ ]:
import os, json, sys, random, importlib.util
from pathlib import Path

IS_KAGGLE = Path('/kaggle').exists()
WORK_DIR = Path(os.environ.get('SIGK_WORK_DIR', '/kaggle/working' if IS_KAGGLE else '/mnt/data/sigk_kaggle_work/local_run')).resolve()
INPUT_DIR = Path(os.environ.get('SIGK_INPUT_DIR', '/kaggle/input' if IS_KAGGLE else str(WORK_DIR / 'input'))).resolve()
WORK_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR = WORK_DIR / 'sigk_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SMOKE_TEST = os.environ.get('SIGK_NOTEBOOK_SMOKE', '0') == '1'
SEED = 42
AUGMENT = 'geometric'
EPOCHS = 18
SAMPLES_PER_EPOCH = 4800
BATCH_SIZE = 16
MODEL_DEPTH = 17
MODEL_FEATURES = 64
PATCH_SIZE = 256
LEARNING_RATE = 3e-5
MIN_LEARNING_RATE = 1e-6
SSIM_WEIGHT = 1.0
EMA_DECAY = 0.999
PSNR_GUARD_DB = 0.15
NUM_WORKERS = 2
USE_AMP = True
MULTI_GPU = False
RUN_TRAINING = True
RUN_BASELINE_TUNING = False
RUN_EVALUATION = True
RUN_LPIPS = True
RUN_TTA = True
EXTRA_SIGMA = 0.02
AUTO_DOWNLOAD_IF_MISSING = True
RUN_NAME = 'dncnn_v3_adamw_bs16_ssim_ema_' + AUGMENT

if SMOKE_TEST:
    EPOCHS, SAMPLES_PER_EPOCH, BATCH_SIZE = 1, 4, 2
    MODEL_DEPTH, MODEL_FEATURES, PATCH_SIZE, NUM_WORKERS = 4, 8, 32, 0
    RUN_LPIPS, EXTRA_SIGMA, RUN_TTA = False, None, True
    RUN_NAME = 'dncnn_v3_smoke'

os.chdir(WORK_DIR)
Path('sigk_denoising').mkdir(exist_ok=True)
Path('sigk_denoising/__init__.py').write_text('"""DnCNN SIGK 2.2 Kaggle."""\n', encoding='utf-8')
print('Kaggle:', IS_KAGGLE, '| INPUT:', INPUT_DIR, '| OUTPUT:', OUTPUT_DIR)
print('V3:', AUGMENT, EPOCHS, 'epok, batch=', BATCH_SIZE, ', patchy/epokę=', SAMPLES_PER_EPOCH)


In [ ]:
import subprocess
required = [('torch','torch'), ('torchvision','torchvision'), ('numpy','numpy'),
            ('PIL','Pillow'), ('skimage','scikit-image'), ('matplotlib','matplotlib'),
            ('tqdm','tqdm'), ('torchmetrics','torchmetrics[image]')]
missing = [pipname for modulename, pipname in required if importlib.util.find_spec(modulename) is None]
if missing:
    print('Instaluję brakujące:', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
import torch, numpy as np
print('PyTorch', torch.__version__, 'CUDA?', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0), 'VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory/1024**3, 1))
    print('Liczba GPU:', torch.cuda.device_count(), '| tryb multi-GPU:', MULTI_GPU)
elif not SMOKE_TEST:
    raise RuntimeError('Włącz GPU w Settings → Accelerator: trening DnCNN na CPU nie jest zalecany.')


In [ ]:
from pathlib import Path
import shutil, urllib.request, zipfile

MANUAL_TRAIN_DIR = None
MANUAL_VALID_DIR = None
DATA_LINKS = {
  'train': 'https://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_train_HR.zip',
  'valid': 'https://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_valid_HR.zip'
}

def scan_files(roots, wanted):
    dirs, archives = [], []
    for base in roots:
        if not base.exists():
            continue
        for folder, subdirs, files in os.walk(base):
            subdirs[:] = [x for x in subdirs if x not in {'.git', '.cache', '__pycache__'}]
            p = Path(folder)
            if p.name.lower() == wanted.lower() and any(p.glob('*.png')):
                dirs.append(p)
            for name in files:
                if name.lower() == f'{wanted.lower()}.zip':
                    archives.append(p / name)
    return sorted(dirs, key=str), sorted(archives, key=str)

def extract_zip(archive, destination):
    destination.mkdir(parents=True, exist_ok=True)
    print('Rozpakowuję:', archive)
    with zipfile.ZipFile(archive) as z:

        root = destination.resolve()
        for member in z.infolist():
            target = (root / member.filename).resolve()
            if target != root and root not in target.parents:
                raise ValueError('Niebezpieczna ścieżka w ZIP: ' + member.filename)
        z.extractall(root)

DATA_ROOT = WORK_DIR / 'div2k_data'
DATA_ROOT.mkdir(parents=True, exist_ok=True)
for split in ('train', 'valid'):
    wanted = f'DIV2K_{split}_HR'
    manual = MANUAL_TRAIN_DIR if split == 'train' else MANUAL_VALID_DIR
    search_roots = [INPUT_DIR, WORK_DIR / 'div2k_data']
    found_dirs, archives = scan_files(search_roots, wanted)
    if manual is not None:
        selected = Path(manual)
        if not selected.is_dir() or not list(selected.glob('*.png')):
            raise FileNotFoundError('Nieprawidłowy katalog: ' + str(selected))
    elif found_dirs:
        selected = found_dirs[0]
    else:
        if archives:
            extract_zip(archives[0], DATA_ROOT / '_archives')
        elif AUTO_DOWNLOAD_IF_MISSING:
            path = DATA_ROOT / (wanted + '.zip')
            print('Pobieram oficjalne DIV2K:', DATA_LINKS[split])
            urllib.request.urlretrieve(DATA_LINKS[split], path)
            extract_zip(path, DATA_ROOT / '_archives')
            path.unlink(missing_ok=True)
        else:
            raise FileNotFoundError(
                f'Brak {wanted}. Użyj Add Input w Kaggle (katalog PNG lub {wanted}.zip) '
                'albo ustaw AUTO_DOWNLOAD_IF_MISSING=True, jeśli masz Internet.'
            )
        found_dirs, _ = scan_files([DATA_ROOT / '_archives'], wanted)
        if not found_dirs:
            raise FileNotFoundError('Nie znaleziono katalogu z PNG po rozpakowaniu ' + wanted)
        selected = found_dirs[0]
    link = DATA_ROOT / wanted
    if not link.exists() and not link.is_symlink():
        try:
            link.symlink_to(selected.resolve(), target_is_directory=True)
        except OSError:

            link.mkdir()
            for file in selected.glob('*.png'):
                shutil.copy2(file, link / file.name)
    png_count = len(list(link.glob('*.png')))
    print(f'{wanted}: {png_count} PNG ← {selected}')
    expected = 800 if split == 'train' else 100
    if png_count != expected and not SMOKE_TEST:
        raise RuntimeError(f'Niekompletny zbiór: {wanted}: {png_count}, oczekiwano {expected}.')

print('DATA_ROOT=', DATA_ROOT)


In [ ]:
%%writefile sigk_denoising/data.py

from __future__ import annotations

import random
from pathlib import Path

import numpy as np
import torch
from PIL import Image, ImageEnhance, ImageOps
from torch.utils.data import Dataset

SIGMAS = (0.01, 0.03)
VALID_IDS = tuple(range(801, 901))
VAL_IDS = tuple(range(801, 851))
TEST_IDS = tuple(range(851, 901))


def find_split_directory(root: str | Path, split: str) -> Path:

    base = Path(root).expanduser()
    actual = "train" if split == "train" else "valid"
    names = (f"DIV2K_{actual}_HR", actual)
    candidates = [base]
    for prefix in (base, base / "DIV2K", base / "raw" / "DIV2K", base / "data" / "DIV2K",
                   base / "p1" / "data" / "raw" / "DIV2K"):
        candidates += [prefix / n for n in names]
    for candidate in candidates:
        if candidate.is_dir() and any(candidate.glob("*.png")):
            return candidate
    raise FileNotFoundError(
        f"Nie znaleziono obrazow {split} w {base}. Oczekiwano DIV2K_{actual}_HR/*.png. "
        "Uruchom python -m sigk_denoising.download albo sprawdz --data-dir."
    )


def list_images(root: str | Path, split: str) -> list[Path]:
    folder = find_split_directory(root, split)
    paths = sorted(folder.glob("*.png"))
    allowed = None
    if split == "val":
        allowed = set(VAL_IDS)
    elif split == "test":
        allowed = set(TEST_IDS)
    if allowed is not None:
        paths = [p for p in paths if p.stem.isdigit() and int(p.stem) in allowed]
    if not paths:
        raise ValueError(f"Brak obrazow PNG dla split={split} w {folder}")
    return paths


def clean_patch(image: Image.Image, size: int, rng: random.Random,
                training: bool, augment: str) -> torch.Tensor:

    image = image.convert("RGB")
    w, h = image.size
    if min(w, h) < size:
        scale = size / min(w, h)
        image = image.resize((round(w * scale), round(h * scale)), Image.Resampling.BICUBIC)
        w, h = image.size
    if training:
        left = rng.randrange(w - size + 1)
        top = rng.randrange(h - size + 1)
    else:
        left, top = (w - size) // 2, (h - size) // 2
    image = image.crop((left, top, left + size, top + size))

    if training and augment in ("geometric", "full"):
        if rng.random() < .5:
            image = ImageOps.mirror(image)
        if rng.random() < .5:
            image = ImageOps.flip(image)
        image = image.rotate(90 * rng.randrange(4))
    if training and augment == "full":

        image = ImageEnhance.Brightness(image).enhance(rng.uniform(.90, 1.10))
        image = ImageEnhance.Contrast(image).enhance(rng.uniform(.90, 1.10))
        image = ImageEnhance.Color(image).enhance(rng.uniform(.90, 1.10))
    rgb = np.array(image, dtype=np.float32).copy() / 255.0
    return torch.from_numpy(rgb).permute(2, 0, 1)


def corrupt(clean: torch.Tensor, sigma: float, seed: int | None = None) -> torch.Tensor:

    if seed is None:
        noise = torch.randn_like(clean)
    else:
        generator = torch.Generator(device="cpu").manual_seed(int(seed))
        noise = torch.randn(clean.shape, dtype=clean.dtype, generator=generator)
    return (clean + noise * sigma).clamp_(0, 1)


class DIV2KDenoising(Dataset):
    def __init__(self, root: str | Path, split: str, size: int = 256,
                 augment: str = "geometric", samples_per_epoch: int = 6400, seed: int = 42):
        if split not in ("train", "val", "test"):
            raise ValueError("split musi byc train/val/test")
        if augment not in ("none", "geometric", "full"):
            raise ValueError("augment: none/geometric/full")
        if size < 16:
            raise ValueError("size >= 16")
        self.paths = list_images(root, split)
        self.split, self.size, self.augment = split, size, augment
        self.samples_per_epoch = samples_per_epoch
        self.seed = seed
        self.epoch = 0

    def set_epoch(self, epoch: int) -> None:
        self.epoch = epoch

    def __len__(self) -> int:
        return self.samples_per_epoch if self.split == "train" else len(self.paths)

    def __getitem__(self, index: int):
        if index < 0 or index >= len(self):
            raise IndexError(index)


        rng = random.Random(f"{self.seed}|{self.epoch if self.split == 'train' else 0}|{index}")
        path = rng.choice(self.paths) if self.split == "train" else self.paths[index]
        with Image.open(path) as image:
            clean = clean_patch(image, self.size, rng, self.split == "train", self.augment)
        sigma = SIGMAS[rng.randrange(len(SIGMAS))] if self.split == "train" else SIGMAS[0]
        noise_seed = rng.getrandbits(63)
        noisy = corrupt(clean, sigma, noise_seed)
        return noisy, clean, torch.tensor(sigma, dtype=torch.float32)


def fixed_noise(clean: torch.Tensor, sigma: float, seed: int, image_index: int, repeat: int = 0):
    return corrupt(clean, sigma, seed + 100_003 * image_index + 10_000_019 * repeat + int(sigma * 1000))


In [ ]:
%%writefile sigk_denoising/model.py

import torch
from torch import nn


class DnCNN(nn.Module):


    def __init__(self, depth: int = 17, features: int = 64, conditioned: bool = True):
        super().__init__()
        if depth < 3 or features < 4:
            raise ValueError("depth >= 3, features >= 4")
        self.depth, self.features, self.conditioned = depth, features, conditioned
        in_ch = 4 if conditioned else 3
        layers: list[nn.Module] = [nn.Conv2d(in_ch, features, 3, padding=1), nn.ReLU(inplace=True)]
        for _ in range(depth - 2):
            layers += [nn.Conv2d(features, features, 3, padding=1, bias=False), nn.BatchNorm2d(features), nn.ReLU(inplace=True)]
        layers.append(nn.Conv2d(features, 3, 3, padding=1))
        self.net = nn.Sequential(*layers)

        nn.init.zeros_(self.net[-1].weight)
        nn.init.zeros_(self.net[-1].bias)

    def forward(self, noisy: torch.Tensor, sigma: torch.Tensor | float) -> torch.Tensor:
        if self.conditioned:
            if not torch.is_tensor(sigma):
                sigma = torch.tensor(sigma, device=noisy.device, dtype=noisy.dtype)
            sigma = sigma.to(device=noisy.device, dtype=noisy.dtype)
            if sigma.ndim == 0:
                sigma = sigma.expand(noisy.shape[0])
            if sigma.ndim != 1 or sigma.shape[0] != noisy.shape[0]:
                raise ValueError("sigma musi byc skalarem lub tensorem [batch]")
            sigma_map = (sigma / 0.03).view(-1, 1, 1, 1).expand(-1, -1, noisy.shape[2], noisy.shape[3])
            x = torch.cat((noisy, sigma_map), dim=1)
        else:
            x = noisy
        return noisy - self.net(x) * sigma.view(-1, 1, 1, 1)


In [ ]:
%%writefile sigk_denoising/baseline.py

import numpy as np
import torch
from skimage.restoration import denoise_bilateral


def bilateral(noisy: torch.Tensor, sigma: float, sigma_spatial: float = 2.0,
              sigma_color_multiplier: float = 2.0) -> torch.Tensor:

    inp = noisy.detach().cpu().permute(1, 2, 0).numpy().astype(np.float32)
    filtered = denoise_bilateral(inp, sigma_color=sigma_color_multiplier * sigma,
                                 sigma_spatial=sigma_spatial, channel_axis=-1)
    return torch.from_numpy(np.asarray(filtered, dtype=np.float32).copy()).permute(2, 0, 1).clamp(0, 1)


In [ ]:
%%writefile sigk_denoising/train.py

import argparse
import csv
import json
import random
from pathlib import Path

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader
from tqdm import tqdm

from .data import DIV2KDenoising, SIGMAS, fixed_noise
from .model import DnCNN


def resolve_device(name: str) -> torch.device:
    if name != "auto":
        device = torch.device(name)
        if device.type == "cuda" and not torch.cuda.is_available():
            raise RuntimeError("Nie wykryto CUDA. Uzyj --device cpu albo sprawdz sterowniki/PyTorch.")
        return device
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def psnr_per_image(pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    mse = ((pred - target) ** 2).flatten(1).mean(1).clamp_min(1e-12)
    return -10 * torch.log10(mse)


@torch.no_grad()
def validate(model: DnCNN, dataset: DIV2KDenoising, device: torch.device, seed: int) -> dict:
    model.eval()
    totals = {sigma: [] for sigma in SIGMAS}
    for i in range(len(dataset)):
        _, clean, _ = dataset[i]
        for sigma in SIGMAS:
            noisy = fixed_noise(clean, sigma, seed, i).unsqueeze(0).to(device)
            target = clean.unsqueeze(0).to(device)
            prediction = model(noisy, sigma).clamp(0, 1)
            totals[sigma].append(psnr_per_image(prediction, target).item())
    return {f"val_psnr_sigma_{sigma:.2f}": float(np.mean(totals[sigma])) for sigma in SIGMAS}


def run(args: argparse.Namespace):
    if args.epochs < 1 or args.batch_size < 1 or args.samples_per_epoch < 1:
        raise ValueError("epochs, batch_size i samples_per_epoch musza byc > 0")
    if args.patch_size % 8:
        raise ValueError("patch_size powinien byc wielokrotnoscia 8 (standard 256)")
    random.seed(args.seed)
    np.random.seed(args.seed)
    torch.manual_seed(args.seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(args.seed)
    if args.deterministic:
        torch.use_deterministic_algorithms(True)
        if torch.backends.cudnn.is_available():
            torch.backends.cudnn.benchmark = False

    train_ds = DIV2KDenoising(args.data_dir, "train", size=args.patch_size, augment=args.augment,
                             samples_per_epoch=args.samples_per_epoch, seed=args.seed)
    val_ds = DIV2KDenoising(args.data_dir, "val", size=args.patch_size, augment="none", seed=args.seed)
    if not args.allow_incomplete and (len(train_ds.paths) != 800 or len(val_ds.paths) != 50):
        raise RuntimeError(f"DIV2K niekompletny: train={len(train_ds.paths)}/800, val={len(val_ds.paths)}/50. "
                           "Do lokalnych testow uzyj --allow-incomplete.")
    device = resolve_device(args.device)
    model = DnCNN(depth=args.depth, features=args.features, conditioned=True).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=args.epochs)
    start_epoch, best_psnr = 0, float("-inf")
    out = Path(args.output_dir)
    out.mkdir(parents=True, exist_ok=True)

    if args.resume:
        checkpoint = torch.load(args.resume, map_location="cpu", weights_only=True)
        expected = checkpoint["model_config"]
        if (expected["depth"], expected["features"]) != (args.depth, args.features):
            raise ValueError("Przy --resume podaj te same --depth i --features jak w checkpoincie")
        model.load_state_dict(checkpoint["state_dict"])
        optimizer.load_state_dict(checkpoint["optimizer"])
        scheduler.load_state_dict(checkpoint["scheduler"])
        start_epoch, best_psnr = checkpoint["epoch"], checkpoint["best_psnr"]
        if start_epoch >= args.epochs:
            raise ValueError("--epochs musi byc wieksze od liczby epok w checkpoincie")
        print(f"Resume: zaczynam od epoki {start_epoch+1}, najlepszy PSNR {best_psnr:.4f}")

    config = vars(args).copy()
    config["data_dir"] = str(config["data_dir"])
    config["output_dir"] = str(config["output_dir"])
    config["resume"] = str(config["resume"]) if config["resume"] else None
    config["n_training_images"] = len(train_ds.paths)
    config["n_validation_images"] = len(val_ds.paths)
    config["train_ids"] = [p.stem for p in train_ds.paths]
    config["val_ids"] = [p.stem for p in val_ds.paths]
    (out / "config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")
    log_path = out / "history.csv"
    if not args.resume and log_path.exists():
        raise FileExistsError(f"Historia juz istnieje: {log_path}. Wybierz inny --output-dir.")
    columns = ["epoch", "train_mse", "train_normalized_mse", "lr", "val_psnr_sigma_0.01", "val_psnr_sigma_0.03", "val_psnr_mean"]
    with log_path.open("a", encoding="utf-8", newline="") as log:
        writer = csv.DictWriter(log, fieldnames=columns)
        if log_path.stat().st_size == 0:
            writer.writeheader()
        for epoch in range(start_epoch, args.epochs):
            train_ds.set_epoch(epoch)
            gen = torch.Generator().manual_seed(args.seed + epoch)
            loader = DataLoader(train_ds, batch_size=args.batch_size, shuffle=True,
                                num_workers=args.workers, pin_memory=device.type == "cuda", generator=gen)
            model.train()
            running_loss, running_norm_loss, seen = 0.0, 0.0, 0
            for noisy, clean, sigma in tqdm(loader, desc=f"epoch {epoch+1}/{args.epochs}", leave=False):
                noisy, clean, sigma = noisy.to(device), clean.to(device), sigma.to(device)
                optimizer.zero_grad(set_to_none=True)
                result = model(noisy, sigma)
                loss = (((result - clean) / sigma.view(-1, 1, 1, 1)) ** 2).mean()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                running_loss += nn.functional.mse_loss(result.detach(), clean).item() * len(noisy)
                running_norm_loss += loss.item() * len(noisy)
                seen += len(noisy)
            scores = validate(model, val_ds, device, args.seed + 1_000_000)
            score = float(np.mean(list(scores.values())))
            current_lr = optimizer.param_groups[0]["lr"]
            scheduler.step()
            row = dict(epoch=epoch+1, train_mse=running_loss/seen, train_normalized_mse=running_norm_loss/seen, lr=current_lr,
                       val_psnr_mean=score, **scores)
            writer.writerow(row)
            log.flush()
            is_best = score > best_psnr
            if is_best:
                best_psnr = score
            state = dict(
                epoch=epoch+1, best_psnr=best_psnr, model_config={"depth": args.depth, "features": args.features,
                                                                 "conditioned": True},
                state_dict=model.state_dict(), optimizer=optimizer.state_dict(), scheduler=scheduler.state_dict(),
                train_config=config,
            )
            torch.save(state, out / "last.pt")
            if is_best:
                torch.save(state, out / "best.pt")
            print(f"Epoka {epoch+1:>3}: MSE={row['train_mse']:.6f}, val PSNR "
                  f"σ=.01 {scores['val_psnr_sigma_0.01']:.3f} / "
                  f"σ=.03 {scores['val_psnr_sigma_0.03']:.3f}, mean={score:.3f} dB" + (" *BEST*" if is_best else ""))
    print(f"Gotowe: {out/'best.pt'} | historia: {log_path}")


def parser():
    p = argparse.ArgumentParser(description="Trening DnCNN na DIV2K (SIGK 2.2)")
    p.add_argument("--data-dir", type=Path, default=Path("data/DIV2K"))
    p.add_argument("--output-dir", type=Path, default=Path("runs/dncnn_geometric"))
    p.add_argument("--epochs", type=int, default=50)
    p.add_argument("--batch-size", type=int, default=8)
    p.add_argument("--samples-per-epoch", type=int, default=3200)
    p.add_argument("--patch-size", type=int, default=256)
    p.add_argument("--augment", choices=["none", "geometric", "full"], default="geometric")
    p.add_argument("--depth", type=int, default=17)
    p.add_argument("--features", type=int, default=64)
    p.add_argument("--lr", type=float, default=1e-3)
    p.add_argument("--device", default="auto")
    p.add_argument("--workers", type=int, default=0, help="Windows: domyslnie 0 (mozna zwiekszyc)")
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--resume", type=Path, default=None)
    p.add_argument("--deterministic", action="store_true")
    p.add_argument("--allow-incomplete", action="store_true", help="TYLKO smoke-test na malym zbiorze")
    return p


if __name__ == "__main__":
    run(parser().parse_args())


In [ ]:
%%writefile sigk_denoising/finetune_v3.py

from __future__ import annotations
import argparse
import copy
import csv
import json
import random
from pathlib import Path

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from .data import DIV2KDenoising, SIGMAS, fixed_noise
from .model import DnCNN
from .train import resolve_device, psnr_per_image


def get_ssim():
    from torchmetrics.functional.image import structural_similarity_index_measure
    return structural_similarity_index_measure


def adamw_parameters(model: nn.Module):
    decay, no_decay = [], []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue

        (no_decay if param.ndim < 2 or name.endswith('.bias') else decay).append(param)
    return [{'params': decay, 'weight_decay': 1e-4},
            {'params': no_decay, 'weight_decay': 0.0}]


@torch.no_grad()
def ema_update(ema: nn.Module, model: nn.Module, decay: float):
    for ema_param, param in zip(ema.parameters(), model.parameters()):
        ema_param.lerp_(param.detach(), 1 - decay)


    for dst, src in zip(ema.buffers(), model.buffers()):
        dst.copy_(src)


@torch.no_grad()
def validate_v3(model: DnCNN, ds: DIV2KDenoising, device: torch.device, seed: int) -> dict:
    from skimage.metrics import structural_similarity
    model.eval()
    metrics = {sigma: {'psnr': [], 'ssim': []} for sigma in SIGMAS}
    for i in range(len(ds)):
        _, clean, _ = ds[i]
        tgt = clean.unsqueeze(0).to(device)
        for sigma in SIGMAS:
            inp = fixed_noise(clean, sigma, seed, i).unsqueeze(0).to(device)
            pred = model(inp, sigma).float().clamp(0, 1)
            metrics[sigma]['psnr'].append(float(psnr_per_image(pred, tgt).item()))

            arr = pred[0].permute(1, 2, 0).cpu().numpy()
            ref = clean.permute(1, 2, 0).numpy()
            metrics[sigma]['ssim'].append(float(structural_similarity(ref, arr, channel_axis=-1, data_range=1.0)))
    results = {}
    for sigma in SIGMAS:
        for key in ('psnr', 'ssim'):
            results[f'val_{key}_sigma_{sigma:.2f}'] = float(np.mean(metrics[sigma][key]))
    results['val_psnr_mean'] = float(np.mean([results[f'val_psnr_sigma_{s:.2f}'] for s in SIGMAS]))
    results['val_ssim_mean'] = float(np.mean([results[f'val_ssim_sigma_{s:.2f}'] for s in SIGMAS]))
    return results


def train(args):
    if args.batch_size != 16 and not args.allow_incomplete:
        print('UWAGA: batch size ustawiony na', args.batch_size, '(domyslnie V3=16)')
    random.seed(args.seed)
    np.random.seed(args.seed)
    torch.manual_seed(args.seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(args.seed)
    device = resolve_device(args.device)
    if device.type == 'cuda':
        torch.backends.cudnn.benchmark = True
    train_ds = DIV2KDenoising(args.data_dir, 'train', size=args.patch_size, augment=args.augment,
                             samples_per_epoch=args.samples_per_epoch, seed=args.seed)
    val_ds = DIV2KDenoising(args.data_dir, 'val', size=args.patch_size, augment='none', seed=args.seed)
    if not args.allow_incomplete and (len(train_ds.paths) != 800 or len(val_ds.paths) != 50):
        raise RuntimeError('Wymagane pelne DIV2K train=800, val=50; do testow allow_incomplete=True')

    model = DnCNN(depth=args.depth, features=args.features, conditioned=True).to(device)
    start_from = Path(args.init_checkpoint) if args.init_checkpoint else None
    if start_from is None and not args.resume:
        raise ValueError('Fine-tuning V3 wymaga init_checkpoint V2. Mozna uzyc --resume V3.')
    initial = None
    if start_from and not args.resume:
        initial = torch.load(start_from, map_location='cpu', weights_only=True)
        if initial['model_config'] != {'depth': args.depth, 'features': args.features, 'conditioned': True}:
            raise ValueError(f'Inna architektura niz V2: {initial["model_config"]}')
        model.load_state_dict(initial['state_dict'])
        print(f'Zaladowano wagi V2, epoka {initial["epoch"]}, plik: {start_from}')
    ema = copy.deepcopy(model).to(device)
    ema.requires_grad_(False)
    net = model
    if args.multi_gpu and device.type == 'cuda' and torch.cuda.device_count() >= 2:
        net = nn.DataParallel(model)
        print(f'DataParallel na {torch.cuda.device_count()} GPU (bez gwarancji przyspieszenia).')
    else:
        print('Trening na', device)
    opt = torch.optim.AdamW(adamw_parameters(model), lr=args.lr,
                            betas=(0.9, 0.999), eps=1e-8)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=args.epochs, eta_min=args.lr_min)
    use_amp = args.amp and device.type == 'cuda'
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
    ssim = get_ssim()
    out = Path(args.output_dir)
    out.mkdir(parents=True, exist_ok=True)
    history_path = out / 'history.csv'
    start_epoch = 0
    initial_metrics = None
    if args.resume:
        prev = torch.load(args.resume, map_location='cpu', weights_only=True)
        if prev.get('model_config') != {'depth': args.depth, 'features': args.features, 'conditioned': True}:
            raise ValueError('Niezgodny checkpoint resume.')
        model.load_state_dict(prev['state_dict'])
        ema.load_state_dict(prev['ema_state_dict'])
        opt.load_state_dict(prev['optimizer'])
        sch.load_state_dict(prev['scheduler'])
        if prev.get('scaler') and use_amp:
            scaler.load_state_dict(prev['scaler'])
        start_epoch = int(prev['epoch'])
        initial_metrics = prev.get('initial_metrics')
        print(f'Resume V3 z epoki {start_epoch}')
    elif history_path.exists():
        raise FileExistsError(f'Wyniki juz istnieja: {history_path}. Zmien RUN_NAME lub wznow z last.pt.')

    if initial_metrics is None:

        initial_metrics = validate_v3(model, val_ds, device, args.seed + 1_000_000)
        print('Wynik poczatkowy V2 na VAL:', initial_metrics)
    ref_psnr = initial_metrics['val_psnr_mean']

    if not args.resume:
        baseline_state = dict(epoch=0, epoch_v2=initial['epoch'] if initial else None,
            model_config={'depth': args.depth, 'features': args.features, 'conditioned': True},
            state_dict=model.state_dict(), ema_state_dict=ema.state_dict(),
            eval_state_dict=model.state_dict(), eval_weights='raw',
            optimizer=opt.state_dict(), scheduler=sch.state_dict(), scaler=scaler.state_dict(),
            best_metrics=initial_metrics, initial_metrics=initial_metrics,
            train_config={**{k: str(v) if isinstance(v, Path) else v for k,v in vars(args).items()},
                          'batch_size': args.batch_size, 'optimizer_name': 'AdamW'},
            best_score=initial_metrics['val_ssim_mean'])
        torch.save(baseline_state, out / 'best.pt')
    best = torch.load(out / 'best.pt', map_location='cpu', weights_only=True)
    best_ssim = float(best['best_score'])
    train_config = {k: str(v) if isinstance(v, Path) else v for k, v in vars(args).items()}
    train_config['optimizer_name'] = 'AdamW'
    (out / 'config.json').write_text(json.dumps({'args': train_config,
       'initial_val': initial_metrics, 'n_train_images': len(train_ds.paths),
       'n_val_images': len(val_ds.paths),
       'selection': f'mean SSIM on VAL, PSNR drop <= {args.psnr_guard_db} dB vs V2 VAL'},
       indent=2, ensure_ascii=False), encoding='utf-8')
    fields = ['epoch', 'train_mse', 'train_norm_mse', 'train_ssim_loss', 'lr',
              'raw_psnr', 'raw_ssim', 'ema_psnr', 'ema_ssim', 'best_eval', 'best_val_psnr', 'best_val_ssim']
    with history_path.open('a', newline='', encoding='utf-8') as handle:
        writer = csv.DictWriter(handle, fieldnames=fields)
        if handle.tell() == 0:
            writer.writeheader()
        for epoch in range(start_epoch, args.epochs):
            train_ds.set_epoch(epoch)
            loader = DataLoader(train_ds, batch_size=args.batch_size, shuffle=True,
                                num_workers=args.workers, pin_memory=device.type == 'cuda',
                                persistent_workers=args.workers > 0,
                                generator=torch.Generator().manual_seed(args.seed+epoch))
            net.train()
            seen, mse_sum, nmse_sum, ssim_sum = 0, 0., 0., 0.
            for noisy, clean, sigma in tqdm(loader, desc=f'V3 {epoch+1}/{args.epochs}', leave=False):
                noisy = noisy.to(device, non_blocking=True)
                clean = clean.to(device, non_blocking=True)
                sigma = sigma.to(device, non_blocking=True)
                opt.zero_grad(set_to_none=True)
                with torch.autocast(device_type=device.type, enabled=use_amp):
                    restored = net(noisy, sigma)

                restored = restored.float()
                nmse = (((restored-clean.float()) / sigma.float().view(-1,1,1,1))**2).mean()
                ssim_loss = 1.0 - ssim(restored.clamp(0,1), clean.float(), data_range=1.0)
                loss = nmse + args.ssim_weight * ssim_loss
                if not torch.isfinite(loss):
                    raise FloatingPointError('Loss jest NaN/Inf. Wylacz AMP lub zmniejsz LR.')
                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(opt)
                scaler.update()
                ema_update(ema, model, args.ema_decay)
                n=len(noisy)
                seen+=n
                mse_sum += nn.functional.mse_loss(restored.detach(), clean).item()*n
                nmse_sum += nmse.detach().item()*n
                ssim_sum += ssim_loss.detach().item()*n
            raw_metrics = validate_v3(model, val_ds, device, args.seed+1_000_000)
            ema_metrics = validate_v3(ema, val_ds, device, args.seed+1_000_000)
            selected = None
            for candidate_name, metrics in [('raw', raw_metrics), ('ema', ema_metrics)]:
                if (metrics['val_psnr_mean'] >= ref_psnr - args.psnr_guard_db and
                        metrics['val_ssim_mean'] > best_ssim):
                    best_ssim = metrics['val_ssim_mean']
                    selected = (candidate_name, metrics)
            current_lr=opt.param_groups[0]['lr']
            sch.step()
            snapshot=dict(epoch=epoch+1,
                model_config={'depth': args.depth, 'features': args.features, 'conditioned': True},
                state_dict=model.state_dict(), ema_state_dict=ema.state_dict(),
                optimizer=opt.state_dict(), scheduler=sch.state_dict(), scaler=scaler.state_dict(),
                train_config=train_config, initial_metrics=initial_metrics,
                best_score=best_ssim, best_metrics=selected[1] if selected else best.get('best_metrics', initial_metrics),
                eval_weights=selected[0] if selected else None,
            )
            torch.save(snapshot, out / 'last.pt')
            if selected:

                snapshot['eval_state_dict'] = selected[1] and copy.deepcopy(
                    model.state_dict() if selected[0]=='raw' else ema.state_dict())
                torch.save(snapshot, out / 'best.pt')
                best = snapshot
            row={'epoch':epoch+1, 'train_mse':mse_sum/seen,
                 'train_norm_mse':nmse_sum/seen, 'train_ssim_loss':ssim_sum/seen,
                 'lr':current_lr, 'raw_psnr':raw_metrics['val_psnr_mean'],
                 'raw_ssim':raw_metrics['val_ssim_mean'],
                 'ema_psnr':ema_metrics['val_psnr_mean'],
                 'ema_ssim':ema_metrics['val_ssim_mean'],
                 'best_eval':selected[0] if selected else '',
                 'best_val_psnr':best['best_metrics']['val_psnr_mean'],
                 'best_val_ssim':best_ssim}
            writer.writerow(row)
            handle.flush()
            print(f'Epoka {epoch+1:>2} | LR {current_lr:.2e} | '
                  f'raw {row["raw_psnr"]:.3f}/{row["raw_ssim"]:.4f} | '
                  f'EMA {row["ema_psnr"]:.3f}/{row["ema_ssim"]:.4f} | '
                  f'best {row["best_val_psnr"]:.3f}/{row["best_val_ssim"]:.4f} '
                  f'{"*" if selected else ""}')
    return out / 'best.pt'


In [ ]:
%%writefile sigk_denoising/tune_baseline.py

import argparse
import json
from itertools import product
from pathlib import Path

import numpy as np
from skimage.metrics import peak_signal_noise_ratio
from tqdm import tqdm

from .baseline import bilateral
from .data import DIV2KDenoising, SIGMAS, fixed_noise
from .evaluate import tensor_to_hwc


def run(args):
    dataset = DIV2KDenoising(args.data_dir, "val", size=256, augment="none")
    if len(dataset) != 50 and not args.allow_incomplete:
        raise RuntimeError(f"Wymagane 50 obrazow do strojenia baseline'u. Jest: {len(dataset)}")
    if args.limit and not args.allow_incomplete:
        raise ValueError("--limit wymaga --allow-incomplete")
    n = min(len(dataset), args.limit) if args.limit else len(dataset)


    settings = list(product(args.spatial, args.color_factors))
    results = {sigma: {setting: [] for setting in settings} for sigma in SIGMAS}
    for i in tqdm(range(n), desc="Tuning bilateral na VAL"):
        _, clean, _ = dataset[i]
        target = tensor_to_hwc(clean)
        for sigma in SIGMAS:
            noisy = fixed_noise(clean, sigma, args.seed, i)
            for setting in settings:
                pred = bilateral(noisy, sigma, sigma_spatial=setting[0], sigma_color_multiplier=setting[1])
                value = float(peak_signal_noise_ratio(target, tensor_to_hwc(pred), data_range=1.0))
                results[sigma][setting].append(value)
    best = {}
    trials = []
    for sigma in SIGMAS:
        for (spatial, color_factor), values in results[sigma].items():
            trials.append({"sigma": sigma, "sigma_spatial": spatial,
                           "sigma_color_multiplier": color_factor, "val_PSNR": float(np.mean(values))})
        winner = max((entry for entry in trials if entry["sigma"] == sigma), key=lambda row: row["val_PSNR"])
        best[f"sigma_{sigma:.2f}"] = {"sigma_spatial": winner["sigma_spatial"],
                                     "sigma_color_multiplier": winner["sigma_color_multiplier"]}
        print(f"σ={sigma:.2f}: PSNR {winner['val_PSNR']:.3f} dB, "
              f"spatial={winner['sigma_spatial']}, color_factor={winner['sigma_color_multiplier']}")
    output = Path(args.output)
    output.parent.mkdir(parents=True, exist_ok=True)
    output.write_text(json.dumps({"protocol": "validation only: 0801-0850; PSNR optimization",
                                  "n_images": n, "seed": args.seed, "best": best,
                                  "trials": trials}, indent=2), encoding="utf-8")
    print(f"Parametry zapisane: {output}")


def parser():
    p = argparse.ArgumentParser(description="Strojenie bilateralnego baseline'u na walidacji DIV2K")
    p.add_argument("--data-dir", type=Path, default=Path("data/DIV2K"))
    p.add_argument("--output", type=Path, default=Path("runs/bilateral_val_tuned.json"))
    p.add_argument("--spatial", type=float, nargs="+", default=[1.0, 2.0, 3.0])
    p.add_argument("--color-factors", type=float, nargs="+", default=[0.5, 1.0, 1.5, 2.0])
    p.add_argument("--seed", type=int, default=123)
    p.add_argument("--limit", type=int)
    p.add_argument("--allow-incomplete", action="store_true", help="wylacznie test techniczny")
    return p


if __name__ == "__main__":
    run(parser().parse_args())


In [ ]:
%%writefile sigk_denoising/evaluate.py

import argparse
import csv
import json
from collections import defaultdict
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import torch
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
from tqdm import tqdm

from .baseline import bilateral
from .data import DIV2KDenoising, SIGMAS, fixed_noise
from .model import DnCNN
from .train import resolve_device


def build_lpips(device: torch.device):
    try:
        from torchmetrics.image.lpip import LearnedPerceptualImagePatchSimilarity
        metric = LearnedPerceptualImagePatchSimilarity(net_type="alex", normalize=True).to(device).eval()
        return metric
    except Exception as exc:
        raise RuntimeError(
            "LPIPS nie jest dostepny. Zainstaluj 'pip install -e .[lpips]' "
            "i zezwol na pierwsze pobranie wag AlexNet. "
            "Do smoke-testu mozesz uzyc --skip-lpips (nie do raportu na zajecia)."
        ) from exc


def tensor_to_hwc(x: torch.Tensor) -> np.ndarray:
    return x.detach().cpu().permute(1, 2, 0).numpy().astype(np.float32)


def metrics(pred: torch.Tensor, clean: torch.Tensor, lpips_metric, device):
    a = tensor_to_hwc(pred.clamp(0, 1))
    b = tensor_to_hwc(clean)
    result = {
        "PSNR": float(peak_signal_noise_ratio(b, a, data_range=1.0)),
        "SSIM": float(structural_similarity(b, a, channel_axis=-1, data_range=1.0)),
        "LPIPS": None,
    }
    if lpips_metric is not None:
        with torch.no_grad():
            result["LPIPS"] = float(lpips_metric(pred.clamp(0, 1).unsqueeze(0).to(device),
                                                 clean.unsqueeze(0).to(device)).item())
    return result


def save_visuals(rows: list, path: Path):
    if not rows:
        return
    fig, axes = plt.subplots(len(rows), 4, figsize=(14, 3.5 * len(rows)), squeeze=False)
    for row, (title, images) in enumerate(rows):
        axes[row, 0].set_ylabel(title, fontsize=10)
        for col, (name, img) in enumerate(zip(("Zaszumiony", "Bilateral", "DnCNN", "Oryginal"), images)):
            axes[row, col].imshow(tensor_to_hwc(img.clamp(0, 1)))
            axes[row, col].set_xticks([])
            axes[row, col].set_yticks([])
            if row == 0:
                axes[row, col].set_title(name)
    fig.tight_layout()
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=125)
    plt.close(fig)


@torch.no_grad()
def infer_tta8(model, noisy: torch.Tensor, sigma: float, device: torch.device) -> torch.Tensor:

    x = noisy.unsqueeze(0).to(device)
    restored = []
    for flip in (False, True):
        z = x.flip(-1) if flip else x
        for k in range(4):
            transformed = z.rot90(k, dims=(-2, -1))
            pred = model(transformed, sigma).float().rot90(-k, dims=(-2,-1))
            if flip:
                pred = pred.flip(-1)
            restored.append(pred)
    return torch.stack(restored, dim=0).mean(0).squeeze(0).clamp(0, 1).cpu()


def run(args):
    device = resolve_device(args.device)
    checkpoint = torch.load(args.checkpoint, map_location="cpu", weights_only=True)
    model = DnCNN(**checkpoint["model_config"]).to(device)
    model.load_state_dict(checkpoint.get("eval_state_dict", checkpoint["state_dict"]))
    print("Ewaluowane wagi:", checkpoint.get("eval_weights", "raw (V2)"))
    model.eval()
    patch_size = int(checkpoint.get("train_config", {}).get("patch_size", 256))
    if patch_size != 256 and not args.allow_incomplete:
        raise ValueError(f"Checkpoint trenowano na patch_size={patch_size}. "
                         "Raport SIGK wymaga 256x256 (dla smoke-testu --allow-incomplete).")
    ds = DIV2KDenoising(args.data_dir, args.split, size=patch_size, augment="none", seed=args.seed)
    expected = 50 if args.split in ("val", "test") else 800
    if len(ds.paths) != expected and not args.allow_incomplete:
        raise RuntimeError(f"Niekompletny zbior {args.split}: {len(ds.paths)}/{expected}")
    if args.limit is not None and not args.allow_incomplete:
        raise ValueError("--limit wymaga --allow-incomplete (wynik nie jest pelnym testem)")
    total = min(len(ds), args.limit) if args.limit else len(ds)
    lpips_metric = None if args.skip_lpips else build_lpips(device)
    sigmas = list(SIGMAS)
    if args.extra_sigma is not None:
        if not 0 < args.extra_sigma <= 0.1:
            raise ValueError("--extra-sigma musi byc z (0,0.1]")
        if args.extra_sigma not in sigmas:
            sigmas.append(args.extra_sigma)

    out = Path(args.output_dir)
    out.mkdir(parents=True, exist_ok=True)
    tuned = None
    if args.bilateral_config:
        config = json.loads(Path(args.bilateral_config).read_text(encoding="utf-8"))
        tuned = config["best"]
        print(f"Strojony baseline z VAL: {args.bilateral_config}")
    per_image = []
    groups = defaultdict(lambda: defaultdict(list))
    visuals = []
    for i in tqdm(range(total), desc=f"Ewaluacja {args.split}"):
        _, clean, _ = ds[i]
        name = ds.paths[i].stem
        for sigma in sigmas:
            for repeat in range(args.noise_repeats):

                noisy = fixed_noise(clean, sigma, args.seed, i, repeat)
                params = tuned.get(f"sigma_{sigma:.2f}") if tuned else None
                spatial = params["sigma_spatial"] if params else args.bilateral_spatial
                color_factor = params["sigma_color_multiplier"] if params else args.bilateral_color_factor
                filtered = bilateral(noisy, sigma, spatial, color_factor)
                with torch.no_grad():
                    output = model(noisy.unsqueeze(0).to(device), sigma).squeeze(0).clamp(0, 1).cpu()
                outputs = {"Noisy (bez filtra)": noisy, "Bilateral (skimage)": filtered,
                           "DnCNN V3": output}
                if getattr(args, 'tta', False):
                    outputs["DnCNN V3 + TTA8"] = infer_tta8(model, noisy, sigma, device)
                for method, pred in outputs.items():
                    values = metrics(pred, clean, lpips_metric, device)
                    for key, value in values.items():
                        if value is not None:
                            groups[(sigma, method)][key].append(value)
                    per_image.append({"image_id": name, "sigma": sigma, "noise_repeat": repeat,
                                      "method": method, **values})
                if repeat == 0 and i < args.examples:
                    visuals.append((f"{name} / σ={sigma}", [noisy, filtered, output, clean]))


    summary = []
    for sigma in sigmas:
        methods = ("Noisy (bez filtra)", "Bilateral (skimage)", "DnCNN V3")
        if getattr(args, 'tta', False):
            methods += ("DnCNN V3 + TTA8",)
        for method in methods:
            values = groups[(sigma, method)]
            row = dict(sigma=sigma, method=method, images=total, noise_repeats=args.noise_repeats,
                       PSNR=float(np.mean(values["PSNR"])), SSIM=float(np.mean(values["SSIM"])),
                       LPIPS=float(np.mean(values["LPIPS"])) if values["LPIPS"] else None)
            summary.append(row)
            lpips_text = "n/a" if row["LPIPS"] is None else f"{row['LPIPS']:.4f}"
            print(f"σ={sigma:.2f} | {method:22} | PSNR {row['PSNR']:.3f} | "
                  f"SSIM {row['SSIM']:.4f} | LPIPS {lpips_text}")
    report = {
        "dataset": "DIV2K", "split": args.split, "split_protocol": "train: 0001-0800, val: 0801-0850, test: 0851-0900",
        "checkpoint": str(args.checkpoint), "checkpoint_epoch": checkpoint["epoch"],
        "checkpoint_eval_weights": checkpoint.get("eval_weights", "raw"),
        "tta_8_enabled": bool(getattr(args, 'tta', False)),
        "train_config": checkpoint.get("train_config", {}),
        "patch_size": patch_size, "seed": args.seed, "evaluated_image_ids": [p.stem for p in ds.paths[:total]],
        "noise_repeats": args.noise_repeats, "sigma_values": sigmas,
        "bilateral_params": {"sigma_spatial": args.bilateral_spatial,
                             "sigma_color_multiplier": args.bilateral_color_factor,
                             "tuned_on_val": tuned is not None, "per_sigma": tuned},
        "lpips_enabled": not args.skip_lpips, "protocol_complete": total == expected and patch_size == 256 and not args.skip_lpips,
        "results": summary,
    }
    (out / "results.json").write_text(json.dumps(report, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
    for filename, rows, keys in (
        ("results.csv", summary, ["sigma", "method", "images", "noise_repeats", "PSNR", "SSIM", "LPIPS"]),
        ("per_image.csv", per_image, ["image_id", "sigma", "noise_repeat", "method", "PSNR", "SSIM", "LPIPS"]),
    ):
        with (out / filename).open("w", newline="", encoding="utf-8") as fp:
            writer = csv.DictWriter(fp, fieldnames=keys)
            writer.writeheader()
            writer.writerows(rows)
    save_visuals(visuals, out / "examples.png")

    table = [
        "# SIGK P1 — zadanie 2.2: wyniki odszumiania",
        "",
        f"- Dane: DIV2K, protokol train 0001-0800 / val 0801-0850 / test 0851-0900.",
        f"- Raportowany split: {args.split}; obrazow: {total}; patch: {patch_size}x{patch_size}.",
        f"- Trening: augment={checkpoint.get('train_config', {}).get('augment', 'nieznany')}, "
        f"epoka checkpointu={checkpoint['epoch']}.",
        f"- Realizacji szumu na obraz: {args.noise_repeats}, seed: {args.seed}.",
        f"- Baseline: skimage.denoise_bilateral, strojony na walidacji: {tuned is not None}.",
        f"- LPIPS: {'pominiety (RAPORT NIEKOMPLETNY)' if args.skip_lpips else 'AlexNet, normalize=True'}.",
        "",
        "| σ | Metoda | PSNR ↑ | SSIM ↑ | LPIPS ↓ |",
        "|---:|---|---:|---:|---:|",
    ]
    for entry in summary:
        lpips = "—" if entry["LPIPS"] is None else f"{entry['LPIPS']:.4f}"
        table.append(f"| {entry['sigma']:.2f} | {entry['method']} | "
                     f"{entry['PSNR']:.3f} | {entry['SSIM']:.4f} | {lpips} |")
    table.extend(["", "![Wizualizacje metod](examples.png)", "",
                  "Metryki to srednia wynikow liczonych osobno na kazdym obrazie "
                  "(i powtorzeniu szumu). Wersja raportu generowana automatycznie; "
                  "uzupelnij wnioski interpretacyjne po pelnym treningu."])
    (out / "report.md").write_text("\n".join(table) + "\n", encoding="utf-8")
    print(f"Wyniki w: {out.resolve()}")
    if args.skip_lpips:
        print("UWAGA: LPIPS pominięto: wyniki NIE sa kompletne wzgledem wymagan SIGK.")
    return report


def parser():
    p = argparse.ArgumentParser(description="SIGK 2.2: ewaluacja DnCNN vs bilateral, caly split test")
    p.add_argument("--data-dir", type=Path, default=Path("data/DIV2K"))
    p.add_argument("--checkpoint", type=Path, default=Path("runs/dncnn_geometric/best.pt"))
    p.add_argument("--output-dir", type=Path, default=Path("runs/dncnn_geometric/evaluation_test"))
    p.add_argument("--split", choices=["val", "test"], default="test")
    p.add_argument("--noise-repeats", type=int, default=1)
    p.add_argument("--examples", type=int, default=4)
    p.add_argument("--seed", type=int, default=123)
    p.add_argument("--device", default="auto")
    p.add_argument("--bilateral-spatial", type=float, default=2.0)
    p.add_argument("--bilateral-color-factor", type=float, default=2.0)
    p.add_argument("--bilateral-config", type=Path, default=None,
                   help="JSON z tune_baseline (strojenie tylko na walidacji)")
    p.add_argument("--extra-sigma", type=float, default=None,
                   help="opcjonalny OOD test dla nieuczonej wartosci sigma, np. 0.02")
    p.add_argument("--tta", action="store_true", help="D4 TTA, 8 predykcji / obraz")
    p.add_argument("--skip-lpips", action="store_true", help="TYLKO smoke-test; nie do raportu")
    p.add_argument("--limit", type=int, default=None, help="TYLKO smoke-test")
    p.add_argument("--allow-incomplete", action="store_true", help="TYLKO smoke-test")
    return p


if __name__ == "__main__":
    args = parser().parse_args()
    if args.noise_repeats < 1:
        parser().error("--noise-repeats musi byc >= 1")
    run(args)


In [ ]:
import matplotlib.pyplot as plt
from sigk_denoising.data import DIV2KDenoising, list_images, SIGMAS

train_paths = list_images(DATA_ROOT, 'train')
val_paths = list_images(DATA_ROOT, 'val')
test_paths = list_images(DATA_ROOT, 'test')
assert set(train_paths).isdisjoint(val_paths) and set(val_paths).isdisjoint(test_paths)
print('Liczebności train/val/test:', len(train_paths), len(val_paths), len(test_paths))
if not SMOKE_TEST:
    assert (len(train_paths),len(val_paths),len(test_paths)) == (800,50,50)
preview = DIV2KDenoising(DATA_ROOT, 'train', size=PATCH_SIZE,
                         augment=AUGMENT, samples_per_epoch=3, seed=SEED)
fig, axes = plt.subplots(2, 3, figsize=(12,7))
for i in range(3):
    noisy, clean, sigma = preview[i]
    for row, arr in enumerate((clean, noisy)):
        axes[row, i].imshow(arr.permute(1,2,0).numpy().clip(0,1))
        axes[row, i].axis('off')
        axes[row, i].set_title(('Clean / augment' if row==0 else f'Noisy, σ={sigma.item():.2f}'))
plt.tight_layout(); plt.show()


In [ ]:
from argparse import Namespace
import os, zipfile, shutil
import torch
from sigk_denoising import finetune_v3
from sigk_denoising.model import DnCNN

RUN_DIR = OUTPUT_DIR / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)

EXTERNAL_V2_CHECKPOINT = None
EXTERNAL_V3_RESUME = None

def locate_v2():
    if EXTERNAL_V2_CHECKPOINT:
        return Path(EXTERNAL_V2_CHECKPOINT)
    candidates = list(INPUT_DIR.rglob('*.pt')) if INPUT_DIR.exists() else []
    v2 = [p for p in candidates if 'v2' in str(p).lower() and p.name == 'best.pt']
    if v2:
        return sorted(v2)[0]
    if candidates:
        for p in candidates:
            try:
                cp = torch.load(p, map_location='cpu', weights_only=True)
                if cp.get('model_config') == {'depth': MODEL_DEPTH, 'features': MODEL_FEATURES, 'conditioned': True}:
                    return p
            except Exception:
                pass

    for z in INPUT_DIR.rglob('*.zip') if INPUT_DIR.exists() else []:
        if not any(term in z.name.lower() for term in ('dncnn', 'sigk', 'result', 'checkpoint')):
            continue
        with zipfile.ZipFile(z) as archive:
            match = next((n for n in archive.namelist()
                          if n.endswith('/best.pt') and 'v2' in n.lower()), None)
            if match:
                dest = OUTPUT_DIR / 'v2_import' / 'best.pt'
                dest.parent.mkdir(exist_ok=True, parents=True)
                with archive.open(match) as source, dest.open('wb') as target:
                    shutil.copyfileobj(source, target)
                return dest
    return None

if SMOKE_TEST:

    local = OUTPUT_DIR / 'v2_smoke_best.pt'
    mock = DnCNN(depth=MODEL_DEPTH, features=MODEL_FEATURES, conditioned=True)
    torch.save({'epoch': 0, 'model_config': {'depth': MODEL_DEPTH, 'features': MODEL_FEATURES,
                'conditioned': True}, 'state_dict': mock.state_dict()}, local)
    V2_CHECKPOINT = local
else:
    V2_CHECKPOINT = locate_v2()
    if V2_CHECKPOINT is None:
        raise FileNotFoundError('Brak V2 best.pt w Kaggle Add Input. Dodaj checkpoint V2 lub ZIP z poprzednimi wynikami!')
print('Początkowy checkpoint V2:', V2_CHECKPOINT)

checkpoint_path = RUN_DIR / 'best.pt'
resume_path = Path(EXTERNAL_V3_RESUME) if EXTERNAL_V3_RESUME else (RUN_DIR / 'last.pt' if (RUN_DIR / 'last.pt').exists() else None)
if RUN_TRAINING:
    last_cp = torch.load(resume_path, map_location='cpu', weights_only=True) if resume_path else None
    if last_cp and last_cp['epoch'] >= EPOCHS:
        print('V3 ukończony: pomijam trening.')
    else:
        train_args = Namespace(
            data_dir=DATA_ROOT, output_dir=RUN_DIR, epochs=EPOCHS,
            batch_size=BATCH_SIZE, samples_per_epoch=SAMPLES_PER_EPOCH,
            patch_size=PATCH_SIZE, augment=AUGMENT,
            depth=MODEL_DEPTH, features=MODEL_FEATURES, lr=LEARNING_RATE, lr_min=MIN_LEARNING_RATE,
            device='auto', workers=NUM_WORKERS, seed=SEED,
            init_checkpoint=V2_CHECKPOINT, resume=resume_path,
            ssim_weight=SSIM_WEIGHT, ema_decay=EMA_DECAY, psnr_guard_db=PSNR_GUARD_DB,
            amp=USE_AMP, multi_gpu=MULTI_GPU, allow_incomplete=SMOKE_TEST)
        finetune_v3.train(train_args)
if not checkpoint_path.exists():
    raise FileNotFoundError('Brak checkpointu best.pt: ' + str(checkpoint_path))
print('BEST V3:', checkpoint_path)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
history_path = RUN_DIR / 'history.csv'
if history_path.exists():
    history = pd.read_csv(history_path)
    display(history.tail(8))
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for key in ('raw_psnr', 'ema_psnr', 'best_val_psnr'):
        axes[0].plot(history['epoch'], history[key], label=key)
    for key in ('raw_ssim', 'ema_ssim', 'best_val_ssim'):
        axes[1].plot(history['epoch'], history[key], label=key)
    axes[0].set_title('Walidacja PSNR / obie sigma')
    axes[1].set_title('Walidacja SSIM / obie sigma')
    for ax in axes:
        ax.grid(True, alpha=.3)
        ax.set_xlabel('Epoka V3')
        ax.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
from sigk_denoising import tune_baseline as tuning_module
import zipfile, shutil
baseline_json = OUTPUT_DIR / 'bilateral_val_tuned.json'

def import_bilateral_from_input():
    if not INPUT_DIR.exists():
        return False
    for p in INPUT_DIR.rglob('bilateral_val_tuned.json'):
        shutil.copyfile(p, baseline_json)
        return True
    for z in INPUT_DIR.rglob('*.zip'):
        if not any(w in z.name.lower() for w in ('dncnn','sigk','result')):
            continue
        with zipfile.ZipFile(z) as archive:
            if 'bilateral_val_tuned.json' in archive.namelist():
                baseline_json.write_bytes(archive.read('bilateral_val_tuned.json'))
                return True
    return False

found = baseline_json.exists() or import_bilateral_from_input()
if RUN_BASELINE_TUNING or not found:
    print('Strojenie filtra bilateralnego NA WALIDACJI...')
    tune_args = Namespace(data_dir=DATA_ROOT, output=baseline_json,
                          spatial=[1.0, 2.0, 3.0, 4.0],
                          color_factors=[0.35, 0.5, 0.75, 1.0],
                          seed=123, limit=2 if SMOKE_TEST else None,
                          allow_incomplete=SMOKE_TEST)
    tuning_module.run(tune_args)
else:
    print('Używam parametrów filtra dostrojonych wcześniej na VAL.')
print('Bilateral:', baseline_json)


In [ ]:
from sigk_denoising import evaluate as evaluate_module
EVAL_DIR = RUN_DIR / 'evaluation_test'
report = None
if RUN_EVALUATION:
    eval_args = Namespace(
        data_dir=DATA_ROOT, checkpoint=checkpoint_path, output_dir=EVAL_DIR,
        split='test', noise_repeats=1, examples=4, seed=123,
        device='auto', bilateral_spatial=2.0, bilateral_color_factor=2.0,
        bilateral_config=baseline_json if baseline_json.exists() else None,
        extra_sigma=EXTRA_SIGMA, skip_lpips=not RUN_LPIPS, tta=RUN_TTA,
        limit=2 if SMOKE_TEST else None, allow_incomplete=SMOKE_TEST
    )
    report = evaluate_module.run(eval_args)
    print('Kompletny protokół?', report['protocol_complete'])


In [ ]:
from IPython.display import Image as IPImage, display, Markdown
if RUN_EVALUATION:
    scores = pd.read_csv(EVAL_DIR / 'results.csv')
    display(scores)
    display(IPImage(filename=str(EVAL_DIR / 'examples.png')))
    display(Markdown((EVAL_DIR / 'report.md').read_text(encoding='utf-8').split('![Wizualizacje')[0]))
    ref = scores[scores['method'] == 'Noisy (bez filtra)'].set_index('sigma')
    for method in ['DnCNN V3', 'DnCNN V3 + TTA8']:
        if method not in set(scores['method']):
            continue
        net = scores[scores['method'] == method].set_index('sigma')
        delta = (net[['PSNR','SSIM','LPIPS']] - ref[['PSNR','SSIM','LPIPS']]).rename(
            columns={'PSNR':'delta_PSNR_dB', 'SSIM':'delta_SSIM', 'LPIPS':'delta_LPIPS'})
        print(f'Zmiana {method} względem zaszumionego obrazu:')
        display(delta)
        if (delta.loc[[0.01,0.03], 'delta_PSNR_dB'] < 0.20).any():
            print('UWAGA: poprawa PSNR < 0.20 dB dla jednej z wymaganych wartości σ.')


In [ ]:
import zipfile
artifact_zip = OUTPUT_DIR / f'{RUN_NAME}_SIGK_2_2_results.zip'
with zipfile.ZipFile(artifact_zip, 'w', zipfile.ZIP_DEFLATED, compresslevel=3) as z:
    for base in (RUN_DIR,):
        for p in base.rglob('*'):
            if p.is_file() and p.resolve() != artifact_zip.resolve():
                z.write(p, arcname=p.relative_to(OUTPUT_DIR))
    if baseline_json.exists():
        z.write(baseline_json, arcname=baseline_json.name)
print('WYNIKI:', artifact_zip)
print('ROZMIAR ZIP [MB]:', round(artifact_zip.stat().st_size / (1024**2), 2))
